# Parte 1 — Engenharia de Dados: Curadoria e Qualidade na Silver

Aplica os tratamentos das sete tabelas Silver, ajusta os tipos e usa nomes de colunas em português. As tabelas Bronze são usadas como origem. As gravações na Silver usam Delta Overwrite.


## Configuração da camada Silver

Define os schemas de origem e destino com o mesmo catálogo usado nos notebooks anteriores.


In [0]:
# Configura os nomes das camadas para a leitura e a gravação das tabelas.

catalog = "cinedata_medallion"

bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

# Período do JSON fornecido. Os widgets permitem ajustar o calendário na Silver.
from datetime import datetime
from zoneinfo import ZoneInfo

hoje = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
dbutils.widgets.text("data_inicio", "10-01-2026", "Data inicial (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "10-07-2026", "Data final (MM-DD-AAAA)")
inicio_periodo = datetime.strptime(dbutils.widgets.get("data_inicio"), "%m-%d-%Y").date()
fim_periodo = datetime.strptime(dbutils.widgets.get("data_fim"), "%m-%d-%Y").date()
if inicio_periodo > fim_periodo or fim_periodo > hoje:
    raise ValueError("O período deve ter início <= fim <= hoje.")


## Apoio — Funções de checagem de qualidade

dq_check conta as linhas que não atendem à condição. dq_check_unique conta as chaves duplicadas. As funções mostram PASS ou FAIL e guardam o resultado das verificações.


In [0]:
# As funções abaixo registram os resultados das checagens de qualidade.

from datetime import datetime
from pyspark.sql import Row, functions as F

dq_results = []

def dq_check(table_name, check_name, df, condition):
    total = df.count()
    failed = df.filter(~F.coalesce(condition, F.lit(False))).count()
    passed = failed == 0

    dq_results.append(
        Row(
            tabela=table_name,
            verificacao=check_name,
            total_linhas=total,
            quantidade_falhas=failed,
            aprovado=passed,
            verificado_em=datetime.now()
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")
    if not passed:
        raise ValueError(f"Falha de qualidade: {table_name} | {check_name}")
    return passed


def dq_check_unique(table_name, check_name, df, key_cols):
    total = df.count()
    duplicates = (
        df.groupBy(*key_cols)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    passed = duplicates == 0

    dq_results.append(
        Row(
            tabela=table_name,
            verificacao=check_name,
            total_linhas=total,
            quantidade_falhas=duplicates,
            aprovado=passed,
            verificado_em=datetime.now()
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {duplicates} chaves duplicadas")
    if not passed:
        raise ValueError(f"Falha de qualidade: {table_name} | {check_name}")
    return passed


def versao_mais_recente(df, colunas):
    # Não combina máximos de colunas diferentes: conserva uma linha inteira da origem.
    # Se o timestamp empatar, os demais campos do struct desempatam de forma determinística.
    # Um empate não permite descobrir qual alteração aconteceu primeiro no sistema de origem.
    return (
        df.groupBy("id")
        .agg(F.max(F.struct("ingestion_datetime", *colunas)).alias("mais_recente"))
        .select("id", "mais_recente.*")
    )


## 1. silver.tb_info_filmes — Tratamento dos metadados

Mantém a versão mais recente por id de filme usando ingestion_datetime. Converte as datas em três formatos, normaliza e traduz o status, renomeia as colunas e deriva ano_lancamento. Valores não mapeáveis de status ficam como Não Informado.


In [0]:
# O timestamp é o primeiro campo do struct, por isso define a versão mais recente.

from pyspark.sql import functions as F

df_bronze_info_filmes = spark.table(f"{bronze_schema}.tb_movies_info")

df_silver_info_filmes = (
    df_bronze_info_filmes

    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))
    .filter(F.col("id").isNotNull() & (F.col("id") > 0))
    .withColumn("ingestion_datetime", F.col("ingestion_datetime").cast("timestamp"))

    # Agrupa por filme e seleciona o registro mais recente.
    # No mesmo timestamp, os demais campos desempatam; não há ordem de atualização na fonte.
    .groupBy("id")
    .agg(
        F.max(
            F.struct(
                "ingestion_datetime",
                "title",
                "original_title",
                "release_date",
                "runtime",
                "original_language",
                "status",
                "overview",
                "tagline"
            )
        ).alias("mais_recente")
    )
    .select("id", "mais_recente.*")

    # Tipagem e conversão das datas
    .withColumn("runtime", F.expr("try_cast(runtime AS INT)"))
    .withColumn("release_date", F.trim(F.col("release_date").cast("string")))
    .withColumn(
        "release_date",
        F.coalesce(
            F.try_to_timestamp(F.col("release_date"), F.lit("yyyy-MM-dd")),
            F.try_to_timestamp(F.col("release_date"), F.lit("dd/MM/yyyy")),
            F.try_to_timestamp(F.col("release_date"), F.lit("MM-dd-yyyy"))
        ).cast("date")
    )

    # Normalização do status
    .withColumn(
        "status",
        F.trim(
            F.regexp_replace(
                F.regexp_replace(F.lower("status"), r"[-_‐‑‒–—−]+", " "),
                r"\s+", " "
            )
        )
    )

    # Tradução do status
    .withColumn(
        "status",
        F.when(F.col("status") == "released", "Lançado")
         .when(F.col("status") == "post production", "Pós-Produção")
         .when(F.col("status") == "in production", "Em Produção")
         .when(F.col("status") == "planned", "Planejado")
         .when(F.col("status") == "rumored", "Rumores")
         .when(F.col("status") == "canceled", "Cancelado")
         .otherwise("Não Informado")
    )

    .select(
        "id", "title", "original_title", "release_date",
        "runtime", "original_language", "status", "overview", "tagline"
    )

    # Mapeamento das colunas
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")

    # Coluna derivada
    .withColumn("ano_lancamento", F.year("data_lancamento"))
)

dq_check(
    "silver.tb_info_filmes", "id_filme não nulo",
    df_silver_info_filmes, F.col("id_filme").isNotNull()
)

dq_check_unique(
    "silver.tb_info_filmes", "id_filme único",
    df_silver_info_filmes, ["id_filme"]
)

display(df_silver_info_filmes.limit(10))
df_silver_info_filmes.printSchema()


[PASS] silver.tb_info_filmes | id_filme não nulo | 0/97879 linhas falharam
[PASS] silver.tb_info_filmes | id_filme único | 0 chaves duplicadas


id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
47933,Independence Day: Resurgence,Independence Day: Resurgence,2016-06-22,120,en,Lançado,"We always knew they were coming back. Using recovered alien technology, the nations of Earth have collaborated on an immense defense program to protect the planet. But nothing can prepare us for the aliens’ advanced and unprecedented force. Only the ingenuity of a few brave men and women can bring our world back from the brink of extinction.",We had twenty years to prepare. So did they.,2016
50022,Super Troopers 2,Super Troopers 2,2018-04-19,100,en,Lançado,"When an international border dispute arises between the U.S. and Canada, the Super Troopers- Mac, Thorny, Foster, Rabbit and Farva, are called in to set up a new Highway Patrol station in the disputed area.",The mustache rides. Again.,2018
166426,Pirates of the Caribbean: Dead Men Tell No Tales,Pirates of the Caribbean: Dead Men Tell No Tales,2017-05-23,128,en,Lançado,"Thrust into an all-new adventure, a down-on-his-luck Capt. Jack Sparrow feels the winds of ill-fortune blowing even more strongly when deadly ghost sailors led by his old nemesis, the evil Capt. Salazar, escape from the Devil's Triangle. Jack's only hope of survival lies in seeking out the legendary Trident of Poseidon, but to find it, he must forge an uneasy alliance with a brilliant and beautiful astronomer and a headstrong young man in the British navy.",null,2017
191231,Sol Invictus,Sol Invictus,2021-06-18,105,en,Lançado,Far into the future a group of young adults are stranded on a hostile alien planet with no hope of getting home.,null,2021
195590,Collide,Collide,2016-06-10,98,en,Lançado,"An American backpacker gets involved with a ring of drug smugglers as their driver, and winds up on the run from his employers across Cologne’s autobahns.",null,2016
208628,Thugocratie,Voyoucratie,2018-01-31,84,fr,Lançado,"Voyoucratie is FGKO’s new film project. It’s a dramatic fiction about a young gangster’s chaotic path into life crushed by a criminal universe that he can’t escape from. It’s also about a frozen portrait of a youth trapped by a system that leads you to delinquency as the only way to exist. The pitch: Sam, a little bandit, finds himself trapped by a corrupted police officer who forces him to get information about Ali, one of the high crime boss known to be very dangerous.",null,2018
217058,the embryo who came in from the cold,The Embryo Who Came in from the Cold,2020-08-20,90,en,Lançado,5 IVF embryos embark on a journey to find their mother.,null,2020
223702,Sausage Party,Sausage Party,2016-07-11,89,en,Lançado,Frank leads a group of supermarket products on a quest to discover the truth about their existence and what really happens when they become chosen to leave the grocery store.,Always use condiments,2016
233841,i hate myself :),i hate myself :),2017-04-07,56,en,Lançado,"Nebbishy filmmaker Joanna Arnow documents her yearlong relationship with racially charged poet-provocateur James Kepple. What starts out as an uncomfortably intimate portrait of a dysfunctional relationship and protracted mid-twenties adolescence, quickly turns into a complex commentary on societal repression, sexuality and self-confrontation through art.",null,2017
236426,Cleanin' Up the Town: Remembering Ghostbusters,Cleanin' Up the Town: Remembering Ghostbusters,2020-01-07,101,en,Lançado,"\The documentary that tells us the full story from where it all started through to the future of the \""\""Ghostbusters\""\"" franchise.\""""","First in 1984. Again in '89. Still in 2016, bustin' makes us feel good",2020


root
 |-- id_filme: long (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = false)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: integer (nullable = true)



In [0]:
# Grava a tabela de informações dos filmes com os tipos e nomes tratados.

df_silver_info_filmes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

## silver.tb_cotacao_dolar — Série contínua e Forward Fill

Executada antes do financeiro. Conserva o histórico anterior e cria dias até o fim solicitado,
incluindo finais de semana após a última cotação. Cada dia recebe a última cotação positiva
disponível até ele, usando join, groupBy e max(struct), sem funções de janela.


In [0]:
df_bronze_cotacao_dolar = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

df_cotacoes = (
    df_bronze_cotacao_dolar
    .select(
        F.expr("try_cast(dataHoraCotacao AS TIMESTAMP)").alias("data_hora_cotacao"),
        F.expr("try_cast(cotacaoCompra AS DECIMAL(18,6))").alias("cotacao_compra"),
        "ingestion_datetime"
    )
    .filter(
        F.col("data_hora_cotacao").isNotNull()
        & (F.col("cotacao_compra") > 0)
        & (F.to_date("data_hora_cotacao") <= F.lit(fim_periodo))
    )

    .groupBy("data_hora_cotacao")
    .agg(F.max(F.struct("ingestion_datetime", "cotacao_compra")).alias("versao"))
    .select("data_hora_cotacao", F.col("versao.cotacao_compra").alias("cotacao_compra"))
)

primeira_cotacao = df_cotacoes.agg(
    F.min(F.to_date("data_hora_cotacao")).alias("data")
).first()["data"]

if primeira_cotacao is None or primeira_cotacao > inicio_periodo:
    raise ValueError("Falta cotação anterior ou igual ao início. Amplie a consulta na Bronze.")

df_calendario = spark.range(1).select(
    F.explode(F.sequence(
        F.lit(primeira_cotacao), F.lit(fim_periodo), F.expr("INTERVAL 1 DAY")
    )).alias("data_cotacao")
)

df_silver_cotacao_dolar = (
    df_calendario
    .join(
        df_cotacoes,
        F.to_date(df_cotacoes["data_hora_cotacao"]) <= df_calendario["data_cotacao"],
        how="left"
    )
    .groupBy("data_cotacao")
    .agg(F.max(F.struct("data_hora_cotacao", "cotacao_compra")).alias("ultima_cotacao"))
    .select("data_cotacao", F.col("ultima_cotacao.cotacao_compra").alias("cotacao_compra"))
)

dq_check_unique("silver.tb_cotacao_dolar", "uma cotação por dia", df_silver_cotacao_dolar, ["data_cotacao"])
dq_check(
    "silver.tb_cotacao_dolar", "cotação positiva e preenchida",
    df_silver_cotacao_dolar, F.col("cotacao_compra") > 0
)

dias_esperados = (fim_periodo - primeira_cotacao).days + 1
if df_silver_cotacao_dolar.count() != dias_esperados:
    raise ValueError("O calendário de cotações tem dias ausentes.")

display(df_silver_cotacao_dolar.orderBy("data_cotacao"))
df_silver_cotacao_dolar.printSchema()


[PASS] silver.tb_cotacao_dolar | uma cotação por dia | 0 chaves duplicadas
[PASS] silver.tb_cotacao_dolar | cotação positiva e preenchida | 0/7 linhas falharam


data_cotacao,cotacao_compra
2026-10-01,5.207300
2026-10-02,5.223200
2026-10-03,5.223200
2026-10-04,5.223200
2026-10-05,4.985300
2026-10-06,4.969200
2026-10-07,4.992900


root
 |-- data_cotacao: date (nullable = false)
 |-- cotacao_compra: decimal(18,6) (nullable = true)



In [0]:
# Grava a série diária resultante do Forward Fill.

# Gravação na Silver
df_silver_cotacao_dolar.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

## silver.tb_financeiro_filmes — Seleção da cotação

Usa a cotação diária mais recente da Silver, já validada e preenchida até o fim do período.
A mesma taxa é aplicada a todos os filmes; este cálculo não é uma conversão histórica pela data de lançamento.


In [0]:
cotacao = (
    spark.table(f"{silver_schema}.tb_cotacao_dolar")
    .filter(F.col("data_cotacao") <= F.lit(fim_periodo))
    .orderBy(F.col("data_cotacao").desc())
    .select("cotacao_compra")
    .first()
)
if cotacao is None or cotacao["cotacao_compra"] is None:
    raise ValueError("Não há cotação válida na Silver para o período.")

taxa_dolar = cotacao["cotacao_compra"]
print(f"Cotação utilizada: R$ {taxa_dolar} por dólar")


Cotação utilizada: R$ 4.992900 por dólar


## 2. silver.tb_financeiro_filmes — Limpeza e cálculos

Transforma textos de ausência em NULL, remove símbolos de moeda e separadores e converte orçamento e receita para decimal. Valores zerados ou negativos ficam ausentes. Calcula BRL, lucro e margem percentual usando lucro dividido por receita, com proteção contra divisão por zero.


In [0]:
# Os cálculos dependentes permanecem NULL quando faltar orçamento ou receita.

# 2. Leitura da Bronze
df_bronze_financeiro_filmes = spark.table(
    f"{bronze_schema}.tb_movies_financials"
)

df_silver_financeiro_filmes = (
    df_bronze_financeiro_filmes
    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))
    .filter(F.col("id").isNotNull() & (F.col("id") > 0))
)

# 3. Limpeza de orçamento e receita
for coluna in ["budget", "revenue"]:
    df_silver_financeiro_filmes = (
        df_silver_financeiro_filmes
        .withColumn(coluna, F.trim(F.col(coluna).cast("string")))

        # Ausência de dados vira NULL antes da conversão
        .withColumn(
            coluna,
            F.when(
                F.lower(F.col(coluna)).isin(
                    "", "unknown", "não informado", "nao informado",
                    "n/a", "na", "null", "none", "nan", "-", "nenhum"
                ),
                F.lit(None).cast("string")
            ).otherwise(F.col(coluna))
        )

        # Remove símbolos de moeda e espaços
        .withColumn(
            coluna,
            F.upper(
                F.regexp_replace(
                    F.col(coluna), r"(?i)USD|US\$|[$€£]|\s", ""
                )
            )
        )

        # Remove separadores de milhar e padroniza o decimal
        .withColumn(
            coluna,
            F.when(
                F.col(coluna).rlike(r"^-?\d{1,3}(\.\d{3})+(,\d+)?$"),
                F.regexp_replace(
                    F.regexp_replace(F.col(coluna), r"\.", ""), ",", "."
                )
            )
            .when(
                F.col(coluna).rlike(r"^-?\d{1,3}(,\d{3})+(\.\d+)?$"),
                F.regexp_replace(F.col(coluna), ",", "")
            )
            .otherwise(F.regexp_replace(F.col(coluna), ",", "."))
        )

        # Converte para decimal, incluindo M = milhão e K = mil
        .withColumn(
            coluna,
            F.when(
                F.col(coluna).endswith("M"),
                F.expr(
                    f"try_cast(regexp_replace({coluna}, 'M$', '') AS DECIMAL(18,6))"
                ) * 1000000
            )
            .when(
                F.col(coluna).endswith("K"),
                F.expr(
                    f"try_cast(regexp_replace({coluna}, 'K$', '') AS DECIMAL(18,6))"
                ) * 1000
            )
            .otherwise(F.expr(f"try_cast({coluna} AS DECIMAL(18,6))"))
            .cast("decimal(18,2)")
        )

        # Valores zerados ou negativos viram NULL
        .withColumn(
            coluna,
            F.when(F.col(coluna) > 0, F.col(coluna))
        )
    )

# Mantém a última carga de cada filme antes dos cálculos e dos joins da Gold.
df_silver_financeiro_filmes = versao_mais_recente(
    df_silver_financeiro_filmes, ["budget", "revenue"]
)

dq_check_unique(
    "silver.tb_financeiro_filmes", "um registro por filme",
    df_silver_financeiro_filmes, ["id"]
)

# 4. Mapeamento e cálculos
df_silver_financeiro_filmes = (
    df_silver_financeiro_filmes
    .select("id", "budget", "revenue")

    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")

    .withColumn(
        "orcamento_brl",
        (F.col("orcamento_usd") * F.lit(taxa_dolar)).cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        (F.col("receita_usd") * F.lit(taxa_dolar)).cast("decimal(18,2)")
    )

    .withColumn(
        "lucro_usd",
        (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)")
    )

    # Margem = lucro / receita × 100
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd") > 0,
            (F.col("lucro_usd") / F.col("receita_usd")) * 100
        ).cast("decimal(18,2)")
    )
)

# 5. Checagens de qualidade
id_valido = dq_check(
    "silver.tb_financeiro_filmes", "id_filme não nulo",
    df_silver_financeiro_filmes,
    F.col("id_filme").isNotNull()
)

orcamento_valido = dq_check(
    "silver.tb_financeiro_filmes", "orçamento positivo ou NULL",
    df_silver_financeiro_filmes,
    F.col("orcamento_usd").isNull() | (F.col("orcamento_usd") > 0)
)

receita_valida = dq_check(
    "silver.tb_financeiro_filmes", "receita positiva ou NULL",
    df_silver_financeiro_filmes,
    F.col("receita_usd").isNull() | (F.col("receita_usd") > 0)
)

if not (id_valido and orcamento_valido and receita_valida):
    raise ValueError("Há falhas nas checagens. Confira antes de gravar.")

display(df_silver_financeiro_filmes.limit(10))
df_silver_financeiro_filmes.printSchema()


[PASS] silver.tb_financeiro_filmes | um registro por filme | 0 chaves duplicadas
[PASS] silver.tb_financeiro_filmes | id_filme não nulo | 0/99006 linhas falharam
[PASS] silver.tb_financeiro_filmes | orçamento positivo ou NULL | 0/99006 linhas falharam
[PASS] silver.tb_financeiro_filmes | receita positiva ou NULL | 0/99006 linhas falharam


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
315635,175000000.00,880166924.00,873757500.00,4394585434.84,705166924.00,3520827934.84,80.12
299537,152000000.00,1131416446.00,758920800.00,5649049173.23,979416446.00,4890128373.23,86.57
324857,90000000.00,375464627.00,449361000.00,1874657336.15,285464627.00,1425296336.15,76.03
259693,40000000.00,null,199716000.00,null,null,null,null
339846,69000000.00,177856751.00,344510100.00,888020972.07,108856751.00,543510872.07,61.20
404368,175000000.00,529323962.00,873757500.00,2642861609.87,354323962.00,1769104109.87,66.94
223702,null,140705322.00,null,702527602.21,null,null,null
396535,8500000.00,87547518.00,42439650.00,437116002.62,79047518.00,394676352.62,90.29
398978,159000000.00,null,793871100.00,null,null,null,null
341013,30000000.00,100014025.00,149787000.00,499360025.42,70014025.00,349573025.42,70.00


root
 |-- id_filme: long (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_percentual: decimal(18,2) (nullable = true)



In [0]:
# Grava os valores financeiros e as colunas calculadas.

df_silver_financeiro_filmes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

## 3. silver.tb_metricas_engajamento — Limpeza e limites

Normaliza separadores da popularidade e converte notas e votos com segurança.
Antes das conversões, identifica aspas residuais e textos em campos numéricos, indícios de column shift.
Nesses registros, a popularidade fica NULL: um número deslocado, como um ano, não deve entrar no ranking.
Marcadores de ausência como Unknown e N/A não acionam essa regra. O descarte é contado e verificado.


In [0]:
# As checagens validam a conversão e os limites numéricos definidos na atividade.

from pyspark.sql import functions as F

df_bronze_metricas_engajamento = spark.table(
    f"{bronze_schema}.tb_movies_metrics"
)

# Identifica indícios de column shift antes de converter notas e votos.
# Um ano deslocado (ex.: 2020) pode ser numérico, mas não é uma popularidade confiável
# quando as demais métricas contêm nomes/textos. Aspas residuais também indicam corrupção.
df_bronze_metricas_engajamento = df_bronze_metricas_engajamento.withColumn(
    "_contexto_corrompido",
    F.coalesce(F.col("popularity").rlike(r'["\\]'), F.lit(False))
)

for coluna in ["vote_average", "vote_count", "averageRating", "numVotes"]:
    df_bronze_metricas_engajamento = df_bronze_metricas_engajamento.withColumn(
        "_contexto_corrompido",
        F.col("_contexto_corrompido") | F.coalesce(
            F.col(coluna).isNotNull()
            & ~F.lower(F.trim(F.col(coluna))).isin(
                "", "unknown", "não informado", "nao informado", "n/a", "na",
                "null", "none", "nan", "-", "nenhum"
            )
            & F.expr(f"try_cast({coluna} AS DECIMAL(28,6))").isNull(),
            F.lit(False)
        )
    )

# 1. Limpeza da popularidade
df_silver_metricas_engajamento = (
    df_bronze_metricas_engajamento
    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))
    .filter(F.col("id").isNotNull() & (F.col("id") > 0))

    # Remove espaços, aspas e barras de escape
    .withColumn(
        "popularity",
        F.regexp_replace(
            F.trim(F.col("popularity").cast("string")),
            r'[\s"\\]', ""
        )
    )

    # Identifica números para conferir possíveis perdas na conversão
    .withColumn(
        "_popularidade_numerica",
        F.coalesce(
            F.col("popularity").rlike(r"^\+?(\d+([.,]\d+)?|\d{1,3}(\.\d{3})+,\d+|\d{1,3}(,\d{3})+\.\d+)$")
            & ~F.col("_contexto_corrompido"),
            F.lit(False)
        )
    )

    # Padroniza separadores sem transformar 54,522 em 54522
    .withColumn(
        "popularity",
        F.when(
            F.col("popularity").rlike(r"^\+?\d{1,3}(\.\d{3})+,\d+$"),
            F.regexp_replace(
                F.regexp_replace(F.col("popularity"), r"\.", ""), ",", "."
            )
        )
        .when(
            F.col("popularity").rlike(r"^\+?\d{1,3}(,\d{3})+\.\d+$"),
            F.regexp_replace(F.col("popularity"), ",", "")
        )
        .otherwise(F.regexp_replace(F.col("popularity"), ",", "."))
    )

    .withColumn(
        "popularity",
        F.expr("try_cast(popularity AS DECIMAL(18,6))")
    )

    # Popularidade negativa vira NULL; zero permanece válido
    .withColumn(
        "popularity",
        F.when(
            (F.col("popularity") >= 0) & ~F.col("_contexto_corrompido"),
            F.col("popularity")
        )
    )
)

# 2. Conversão segura das notas e validação da escala
for coluna in ["vote_average", "averageRating"]:
    df_silver_metricas_engajamento = (
        df_silver_metricas_engajamento
        .withColumn(
            coluna,
            F.expr(f"try_cast({coluna} AS DECIMAL(18,6))")
        )
        .withColumn(
            coluna,
            F.when(F.col(coluna).between(0, 10), F.col(coluna))
        )
    )

# 3. Conversão segura das contagens de votos
for coluna in ["vote_count", "numVotes"]:
    df_silver_metricas_engajamento = (
        df_silver_metricas_engajamento
        .withColumn(coluna, F.trim(F.col(coluna).cast("string")))

        # Aceita inteiros e valores como 100.0, sem truncar 100.5
        .withColumn(
            coluna,
            F.when(
                F.col(coluna).rlike(r"^[+-]?\d+(\.0+)?$"),
                F.expr(
                    f"try_cast(try_cast({coluna} AS DECIMAL(20,0)) AS BIGINT)"
                )
            )
        )

        # Contagem negativa vira NULL
        .withColumn(
            coluna,
            F.when(F.col(coluna) >= 0, F.col(coluna))
        )
    )

# Explicita o descarte de popularidade em contextos corrompidos; não estima o valor real.
qtd_contextos_corrompidos = df_silver_metricas_engajamento.filter(
    F.col("_contexto_corrompido")
).count()
print(f"Registros com contexto corrompido de métricas: {qtd_contextos_corrompidos}")

dq_check(
    "silver.tb_metricas_engajamento", "popularidade de contexto corrompido fica NULL",
    df_silver_metricas_engajamento,
    ~F.col("_contexto_corrompido") | F.col("popularity").isNull()
)

# Depois de limpar, mantém uma linha inteira da carga mais recente por filme.
df_silver_metricas_engajamento = versao_mais_recente(
    df_silver_metricas_engajamento,
    ["popularity", "vote_average", "vote_count", "averageRating", "numVotes", "_popularidade_numerica"]
)
dq_check_unique(
    "silver.tb_metricas_engajamento", "um registro por filme",
    df_silver_metricas_engajamento, ["id"]
)

# 4. Mapeamento das colunas
df_silver_metricas_engajamento = (
    df_silver_metricas_engajamento
    .select(
        "id", "popularity", "vote_average", "vote_count",
        "averageRating", "numVotes", "_popularidade_numerica"
    )
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
)

# 5. Checagens de qualidade
validacoes = [
    dq_check(
        "silver.tb_metricas_engajamento", "id_filme não nulo",
        df_silver_metricas_engajamento,
        F.col("id_filme").isNotNull()
    ),

    dq_check(
        "silver.tb_metricas_engajamento", "popularidade numérica foi convertida",
        df_silver_metricas_engajamento,
        ~F.col("_popularidade_numerica") | F.col("popularidade").isNotNull()
    ),

    dq_check(
        "silver.tb_metricas_engajamento", "nota TMDB entre 0 e 10 ou NULL",
        df_silver_metricas_engajamento,
        F.col("nota_media_tmdb").isNull()
        | F.col("nota_media_tmdb").between(0, 10)
    ),

    dq_check(
        "silver.tb_metricas_engajamento", "nota IMDb entre 0 e 10 ou NULL",
        df_silver_metricas_engajamento,
        F.col("nota_media_imdb").isNull()
        | F.col("nota_media_imdb").between(0, 10)
    )
]

for coluna in ["popularidade", "qtd_votos_tmdb", "qtd_votos_imdb"]:
    validacoes.append(
        dq_check(
            "silver.tb_metricas_engajamento",
            f"{coluna} não negativo ou NULL",
            df_silver_metricas_engajamento,
            F.col(coluna).isNull() | (F.col(coluna) >= 0)
        )
    )

if not all(validacoes):
    raise ValueError("Há falhas nas checagens. Confira antes de gravar.")

# Remove a coluna temporária da checagem
df_silver_metricas_engajamento = (
    df_silver_metricas_engajamento
    .drop("_popularidade_numerica")
)

display(df_silver_metricas_engajamento.limit(10))
df_silver_metricas_engajamento.printSchema()


Registros com contexto corrompido de métricas: 42530
[PASS] silver.tb_metricas_engajamento | popularidade de contexto corrompido fica NULL | 0/1073640 linhas falharam
[PASS] silver.tb_metricas_engajamento | um registro por filme | 0 chaves duplicadas
[PASS] silver.tb_metricas_engajamento | id_filme não nulo | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | popularidade numérica foi convertida | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | nota TMDB entre 0 e 10 ou NULL | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | nota IMDb entre 0 e 10 ou NULL | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | popularidade não negativo ou NULL | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | qtd_votos_tmdb não negativo ou NULL | 0/99013 linhas falharam
[PASS] silver.tb_metricas_engajamento | qtd_votos_imdb não negativo ou NULL | 0/99013 linhas falharam


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
315635,65.880000,7.345000,20507,7.400000,835116
299537,50.399000,6.843000,14657,null,671609
324857,null,8.404000,null,null,null
259693,80.980000,7.293000,7769,7.300000,348713
339846,45.123000,6.092000,7545,5.500000,221366
404368,49.649000,7.200000,6999,7.000000,208886
223702,44.804000,5.734000,6789,6.100000,229032
396535,49.553000,7.800000,null,7.600000,314257
398978,23.095000,7.615000,6203,7.800000,479040
341013,19.968000,6.400000,5860,6.700000,221894


root
 |-- id_filme: long (nullable = true)
 |-- popularidade: decimal(18,6) (nullable = true)
 |-- nota_media_tmdb: decimal(18,6) (nullable = true)
 |-- qtd_votos_tmdb: long (nullable = true)
 |-- nota_media_imdb: decimal(18,6) (nullable = true)
 |-- qtd_votos_imdb: long (nullable = true)



In [0]:
# Grava apenas as colunas finais de métricas, com nomes em português.

df_silver_metricas_engajamento.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

## 4. silver.tb_avaliacoes_usuarios — Tratamento das avaliações

Remove duplicatas idênticas por filme, usuário, nota e comentário antes das conversões. Notas fora de 0 a 10 ficam NULL. Comentários ausentes ou formados apenas por espaços recebem Sem comentário.


In [0]:
# A deduplicação considera a combinação original das quatro colunas da avaliação.

from pyspark.sql import functions as F

df_bronze_avaliacoes_usuarios = spark.table(
    f"{bronze_schema}.tb_movies_reviews"
)

# Remove somente avaliações idênticas na origem
df_silver_avaliacoes_usuarios = (
    df_bronze_avaliacoes_usuarios
    .select("id", "nome", "nota", "comentario")
    .dropDuplicates(["id", "nome", "nota", "comentario"])
)

# Confere a deduplicação antes das conversões
dq_check_unique(
    "silver.tb_avaliacoes_usuarios",
    "sem avaliações idênticas na origem",
    df_silver_avaliacoes_usuarios,
    ["id", "nome", "nota", "comentario"]
)

df_silver_avaliacoes_usuarios = (
    df_silver_avaliacoes_usuarios

    # Tipagem
    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))
    .filter(F.col("id").isNotNull() & (F.col("id") > 0))
    .withColumn("nota", F.expr("try_cast(nota AS DOUBLE)"))

    # Notas fora de 0 a 10 ficam NULL
    .withColumn(
        "nota",
        F.when(F.col("nota").between(0, 10), F.col("nota"))
    )

    # Preenche comentários ausentes ou compostos apenas por espaços
    .withColumn(
        "comentario",
        F.when(
            F.col("comentario").isNull()
            | F.col("comentario").rlike(r"^\s*$"),
            "Sem comentário"
        ).otherwise(F.col("comentario"))
    )

    # Mapeamento das colunas
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")

    # Valores diferentes na origem podem ficar iguais após a limpeza (ex.: "" e " ").
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)

dq_check_unique(
    "silver.tb_avaliacoes_usuarios", "avaliações finais sem duplicatas",
    df_silver_avaliacoes_usuarios,
    ["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"]
)

# Checagens dos tratamentos
dq_check(
    "silver.tb_avaliacoes_usuarios",
    "nota entre 0 e 10 ou NULL",
    df_silver_avaliacoes_usuarios,
    F.col("nota_usuario").isNull()
    | F.col("nota_usuario").between(0, 10)
)

dq_check(
    "silver.tb_avaliacoes_usuarios",
    "comentário preenchido",
    df_silver_avaliacoes_usuarios,
    F.col("comentario_usuario").isNotNull()
    & ~F.col("comentario_usuario").rlike(r"^\s*$")
)

display(df_silver_avaliacoes_usuarios.limit(10))
df_silver_avaliacoes_usuarios.printSchema()


[PASS] silver.tb_avaliacoes_usuarios | sem avaliações idênticas na origem | 0 chaves duplicadas
[PASS] silver.tb_avaliacoes_usuarios | avaliações finais sem duplicatas | 0 chaves duplicadas
[PASS] silver.tb_avaliacoes_usuarios | nota entre 0 e 10 ou NULL | 0/32412 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | comentário preenchido | 0/32412 linhas falharam


id_filme,nome_usuario,nota_usuario,comentario_usuario
662277,Henrique Ferreira 463,9.2,"Adorei cada minuto, uma experiência inesquecível."
558208,Gabriela Pinto 328,3.6,"Não gostei, história confusa."
1200005,Gabriel Ferreira 436,1.5,"Terrível, um dos piores que já vi."
1321549,Carlos Souza 197,0.2,Não recomendo de jeito nenhum.
932481,Paulo Barbosa,9.1,Sem comentário
874124,Natália Ribeiro 337,8.5,Excepcional! História envolvente e atuações impecáveis.
812869,Gustavo Alves 208,1.1,Péssimo em todos os sentidos.
664847,Paulo Dias 352,0.2,Horrível! Perda de tempo.
648316,Marcio Rocha 386,9.5,Perfeito! Um dos melhores filmes que já vi.
545577,Camila Fernandes,1.0,Não recomendo de jeito nenhum.


root
 |-- id_filme: long (nullable = true)
 |-- nome_usuario: string (nullable = true)
 |-- nota_usuario: double (nullable = true)
 |-- comentario_usuario: string (nullable = true)



In [0]:
# Grava as avaliações com as colunas renomeadas e os tratamentos aplicados.

# Gravação na Silver
df_silver_avaliacoes_usuarios.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

## 5. silver.tb_generos — Desmembramento e limpeza

Padroniza vírgula, ponto e vírgula e barra vertical antes do split. Explode cria uma linha por gênero do filme. Remove ruídos nas extremidades e conserva os valores da lista de gêneros válidos.


In [0]:
# A lista de gêneros válidos permite excluir números e textos deslocados desse campo.

generos_validos = [
    "action", "adventure", "animation", "comedy", "crime",
    "documentary", "drama", "family", "fantasy", "history",
    "horror", "music", "mystery", "romance", "science fiction",
    "tv movie", "thriller", "war", "western"
]

df_bronze_creditos = spark.table(
    f"{bronze_schema}.tb_credits_and_tags"
)

df_silver_generos = (
    df_bronze_creditos
    .select("id", "genres")

    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))
    .filter(F.col("id").isNotNull() & (F.col("id") > 0))

    # Padroniza os separadores antes do split
    .withColumn(
        "genres",
        F.regexp_replace(F.col("genres"), r"[;|]", ",")
    )

    # Separa os elementos e cria uma linha para cada gênero
    .withColumn(
        "genero",
        F.explode(F.split(F.col("genres"), ","))
    )

    # Remove espaços, aspas e barras residuais nas extremidades
    .withColumn(
        "genero",
        F.trim(
            F.regexp_replace(
                F.col("genero"),
                r'^[\s"\\]+|[\s"\\]+$',
                ""
            )
        )
    )

    # Mantém somente valores pertencentes ao domínio de gêneros
    .filter(F.lower(F.col("genero")).isin(generos_validos))
    .withColumn("genero", F.initcap(F.lower(F.col("genero"))))

    .select("id", "genero")
    .withColumnRenamed("id", "id_filme")
    .dropDuplicates(["id_filme", "genero"])
)

# Checagem
dq_check(
    "silver.tb_generos",
    "gênero pertence ao domínio permitido",
    df_silver_generos,
    F.lower(F.col("genero")).isin(generos_validos)
)

display(df_silver_generos.limit(10))
df_silver_generos.printSchema()


[PASS] silver.tb_generos | gênero pertence ao domínio permitido | 0/142160 linhas falharam


id_filme,genero
284052,Action
398818,Romance
339846,Action
333371,Thriller
508439,Fantasy
603692,Crime
278154,Adventure
385687,Thriller
180863,Drama
68730,Drama


root
 |-- id_filme: long (nullable = true)
 |-- genero: string (nullable = false)



In [0]:
# Grava os gêneros desmembrados, com um gênero em cada linha.

# Gravação na Silver
df_silver_generos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_generos")


## silver.tb_pessoas_empresas — Consolidação das entidades

Desmembra cast, directors, writers e production_companies e classifica Ator, Diretor,
Roteirista e Produtora. Padroniza os nomes, elimina duplicatas, marcadores de ausência,
números e resíduos evidentes de column shift. Linhas cujo contexto já está corrompido
são desconsideradas nesta tabela; continuam preservadas na Bronze.

As regras de comprimento e quantidade de palavras são conservadoras: uma origem corrompida
não permite recuperar todos os créditos com certeza. Confira os descartes se houver interesse
em nomes longos, nomes artísticos ou produtoras com pontuação incomum.


In [0]:
# A combinação de filme, nome e tipo preserva atuações diferentes da mesma pessoa.

from pyspark.sql import functions as F

df_bronze_creditos = spark.table(
    f"{bronze_schema}.tb_credits_and_tags"
)

# Padroniza os separadores das listas
df_creditos = (
    df_bronze_creditos
    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))
    .filter(F.col("id").isNotNull() & (F.col("id") > 0))
)

# Se genres contém uma sinopse ou um número, as posições seguintes não são confiáveis.
# Campos de gênero ausentes são permitidos: a ausência de gênero não invalida os créditos.
dominio_generos = "|".join(generos_validos)
padrao_generos = rf"(?i)^\s*({dominio_generos})(\s*[,;|]\s*({dominio_generos}))*\s*$"
df_creditos = df_creditos.filter(
    F.col("genres").isNull()
    | F.lower(F.trim(F.col("genres"))).isin("", "nenhum", "n/a", "[]", "unknown", "não informado")
    | F.col("genres").rlike(padrao_generos)
)

# Resíduos de aspas/barras, URLs e números em país/idioma sinalizam column shift.
for coluna in ["production_countries", "spoken_languages"]:
    df_creditos = df_creditos.filter(
        F.col(coluna).isNull()
        | F.lower(F.trim(F.col(coluna))).isin("", "n/a", "nenhum", "[]", "unknown", "não informado")
        | (~F.col(coluna).rlike(r'["\\/0-9]'))
    )

for coluna in ["cast", "directors", "writers", "production_companies"]:
    df_creditos = df_creditos.withColumn(
        coluna,
        F.regexp_replace(F.col(coluna), r"[;|]", ",")
    )

# Atores
df_atores = (
    df_creditos
    .select(
        "id",
        F.explode(F.split(F.col("cast"), ",")).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Ator"))
)

# Diretores
df_diretores = (
    df_creditos
    .select(
        "id",
        F.explode(F.split(F.col("directors"), ",")).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Diretor"))
)

# Roteiristas
df_roteiristas = (
    df_creditos
    .select(
        "id",
        F.explode(F.split(F.col("writers"), ",")).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Roteirista"))
)

# Produtoras
df_produtoras = (
    df_creditos
    .select(
        "id",
        F.explode(
            F.split(F.col("production_companies"), ",")
        ).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Produtora"))
)

# Consolidação e padronização
df_silver_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)

    .withColumn(
        "nome_entidade",
        F.initcap(F.lower(F.trim(
            F.regexp_replace(F.col("nome_entidade"), r"\s+", " ")
        )))
    )

    # Remove elementos vazios gerados pelas listas
    .filter(
        F.col("nome_entidade").isNotNull()
        & ~F.col("nome_entidade").rlike(r"^\s*$")
    )

    .withColumnRenamed("id", "id_filme")

    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

# Marcadores e atributos de contexto não são nomes de pessoas ou produtoras.
termos_fora_contexto = generos_validos + [
    "", "[]", "{}", "nenhum", "n/a", "na", "null", "none", "nan", "unknown",
    "não informado", "nao informado", "-", "english", "french", "spanish", "german",
    "portuguese", "italian", "japanese", "mandarin", "russian", "hindi", "no language",
    "united states of america", "united kingdom", "france", "germany", "japan", "brazil"
]

df_silver_pessoas_empresas = (
    df_silver_pessoas_empresas
    .filter(~F.lower(F.col("nome_entidade")).isin(termos_fora_contexto))

    # Exclui caminhos, URLs, aspas corrompidas e fragmentos claros de frases.
    .filter(~F.col("nome_entidade").rlike(r'["\\/!?]'))
    .filter(F.length(F.col("nome_entidade")) <= 120)
    .filter(F.size(F.split(F.col("nome_entidade"), " ")) <= 12)

    # Números deslocados não são pessoas. Para produtoras, 42 e 1091 são nomes
    # encontrados em linhas contextualizadas da base; não se apagam esses nomes.
    .filter(
        ~F.col("nome_entidade").rlike(r"^[+-]?\d+([.,]\d+)?$")
        | (
            (F.col("tipo_entidade") == "Produtora")
            & F.col("nome_entidade").isin("42", "1091")
        )
    )
)

# Checagens
dq_check_unique(
    "silver.tb_pessoas_empresas",
    "associações sem duplicatas",
    df_silver_pessoas_empresas,
    ["id_filme", "nome_entidade", "tipo_entidade"]
)

dq_check(
    "silver.tb_pessoas_empresas",
    "tipo de entidade válido",
    df_silver_pessoas_empresas,
    F.col("tipo_entidade").isin(
        "Ator", "Diretor", "Roteirista", "Produtora"
    )
)

dq_check(
    "silver.tb_pessoas_empresas",
    "nome da entidade preenchido",
    df_silver_pessoas_empresas,
    F.col("nome_entidade").isNotNull()
    & ~F.col("nome_entidade").rlike(r"^\s*$")
)

display(df_silver_pessoas_empresas.limit(10))
df_silver_pessoas_empresas.printSchema()


[PASS] silver.tb_pessoas_empresas | associações sem duplicatas | 0 chaves duplicadas
[PASS] silver.tb_pessoas_empresas | tipo de entidade válido | 0/879872 linhas falharam
[PASS] silver.tb_pessoas_empresas | nome da entidade preenchido | 0/879872 linhas falharam


id_filme,nome_entidade,tipo_entidade
166426,Kaya Scodelario,Ator
458156,Ian Mcshane,Ator
336843,Ki Hong Lee,Ator
291870,Henry Zebrowski,Ator
576845,Aimée Cassettari,Ator
297802,Dolph Lundgren,Ator
293167,Brie Larson,Ator
508943,Jim Gaffigan,Ator
338970,Dominic West,Ator
43074,Chris Hemsworth,Ator


root
 |-- id_filme: long (nullable = true)
 |-- nome_entidade: string (nullable = false)
 |-- tipo_entidade: string (nullable = false)



In [0]:
# Grava a tabela unificada com os quatro tipos de entidade.

# Gravação na Silver
df_silver_pessoas_empresas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")